
# S — Control transition count \(N\) without redefining \(H\)

This audit asks a narrower question than the previous \(H/N\) normalization:

> **After controlling the number of supported transitions \(N\), how much of the single–full climate ranking agreement remains?**

The original cumulative metric \(H\) is **not** redefined.

Three complementary analyses are used.

### A. Exact-\(N\) conditional pair concordance
Only climate pairs with exactly the same supported-transition count are compared.

For climate pair \(i,j\),

\[
N_i=N_j,
\]

and the ordering induced by \(H_{\rm single}\) is compared with the ordering induced by \(H_{\rm full}\).

This is the cleanest occupancy control because \(N\) is held exactly constant.

### B. Caliper-matched \(N\) concordance
For sensitivity, pairs are retained only when

\[
\frac{|N_i-N_j|}
{\tfrac12(N_i+N_j)}
\le \delta,
\]

with \(\delta=1,\,2.5,\,5,\,10\%\).

### C. Partial Spearman correlation controlling \(N\)
Spearman ranks are used and the standard partial-correlation identity is applied:

\[
\rho_{FS\cdot N}
=
\frac{
\rho_{FS}-\rho_{FN}\rho_{SN}
}{
\sqrt{(1-\rho_{FN}^2)(1-\rho_{SN}^2)}
}.
\]

This keeps the cumulative \(H\) values intact and asks whether the residual rank relationship between single and full remains after conditioning on transition count.

The three analyses answer related but not identical questions; exact-\(N\) concordance is treated as the primary result.


In [ ]:

from google.colab import drive
from google.colab import files as colab_files
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
preferred = DRIVE/'climate_screening_v1_3_1_uncertainty_audit'/'climate_screening_backbone'

if preferred.exists():
    PROJECT = preferred
else:
    hits = list(DRIVE.rglob('climate_screening_backbone'))
    PROJECT = hits[0] if hits else None

if PROJECT is not None:
    QDIR = PROJECT/'results/Q_confirmatory_nested_uncertainty_timescale'
    OUT = PROJECT/'results/S_controlled_N_rank_agreement'
    OUT.mkdir(parents=True,exist_ok=True)
else:
    QDIR = None
    OUT = Path('/content/S_controlled_N_rank_agreement')
    OUT.mkdir(parents=True,exist_ok=True)

# ------------------------------------------------------------
# Find Q climate-level output; upload fallback
# ------------------------------------------------------------
SCORES = None

if QDIR is not None:
    p = QDIR/'nested_climate_scores.csv'
    if p.exists():
        SCORES = p

if SCORES is None:
    print('nested_climate_scores.csv was not found in Drive.')
    print('Upload Q output: nested_climate_scores.csv')
    up = colab_files.upload()

    exact = [n for n in up if n == 'nested_climate_scores.csv']
    if exact:
        SCORES = Path('/content')/exact[0]
    else:
        candidates = [
            n for n in up
            if n.lower().endswith('.csv')
            and 'nested_climate_scores' in Path(n).stem
        ]
        if not candidates:
            raise FileNotFoundError('nested_climate_scores.csv is required.')
        SCORES = Path('/content')/candidates[0]

scores = pd.read_csv(SCORES)

required = {
    'material','branch','temporal_mode','representation',
    'climate_class','city','n_transitions','H','H_bal'
}
missing = required - set(scores.columns)
if missing:
    raise ValueError(f'Missing columns in scores: {sorted(missing)}')

print('SCORES:',SCORES)
print('rows:',len(scores))
print('materials:',scores.material.nunique())
display(scores.head())


## 1. Build one matched climate table per material × branch × temporal mode

In [ ]:

def matched_climate_table(g, metric):
    # n_transitions should be the same across representations because Q used
    # a common transition mask. We nevertheless verify it explicitly.
    idx = ['climate_class','city']

    f = (
        g[g.representation=='full']
        [idx + ['n_transitions',metric]]
        .rename(columns={'n_transitions':'N_full',metric:'full'})
    )

    s = (
        g[g.representation=='single_ref']
        [idx + ['n_transitions',metric]]
        .rename(columns={'n_transitions':'N_single',metric:'single'})
    )

    z = f.merge(s,on=idx,how='inner').dropna()

    if len(z):
        z['N_equal_across_rep'] = z.N_full.eq(z.N_single)
        if not z.N_equal_across_rep.all():
            raise ValueError(
                'Q common-support invariant failed: '
                'n_transitions differs between full and single.'
            )
        z['N'] = z.N_full.astype(int)

    return z


## 2. Conditional pairwise concordance kernel

In [ ]:

EPS = 1e-12
CALIPERS = [0.0,0.01,0.025,0.05,0.10]

def conditional_pair_metrics(full, single, N, rel_caliper=0.0):
    f = np.asarray(full,float)
    s = np.asarray(single,float)
    n = np.asarray(N,float)

    k = len(f)
    if k < 2:
        return {
            'n_candidate_pairs':0,
            'n_comparable_pairs':0,
            'conditional_tau':np.nan,
            'concordance':np.nan,
            'discordance':np.nan
        }

    i,j = np.triu_indices(k,1)

    if rel_caliper == 0:
        keepN = n[i] == n[j]
    else:
        rel = np.abs(n[i]-n[j]) / (
            0.5*(np.abs(n[i])+np.abs(n[j])) + EPS
        )
        keepN = rel <= rel_caliper

    i = i[keepN]
    j = j[keepN]

    if len(i)==0:
        return {
            'n_candidate_pairs':0,
            'n_comparable_pairs':0,
            'conditional_tau':np.nan,
            'concordance':np.nan,
            'discordance':np.nan
        }

    df = f[i]-f[j]
    ds = s[i]-s[j]

    # Kendall-like comparison: exclude ties in either ordering.
    comparable = (df != 0) & (ds != 0)
    df = df[comparable]
    ds = ds[comparable]

    if len(df)==0:
        return {
            'n_candidate_pairs':len(i),
            'n_comparable_pairs':0,
            'conditional_tau':np.nan,
            'concordance':np.nan,
            'discordance':np.nan
        }

    concord = np.sign(df) == np.sign(ds)
    p_conc = float(np.mean(concord))
    p_disc = 1.0-p_conc

    # When ties are excluded, this is (C-D)/(C+D).
    tau_cond = p_conc-p_disc

    return {
        'n_candidate_pairs':len(i),
        'n_comparable_pairs':len(df),
        'conditional_tau':tau_cond,
        'concordance':p_conc,
        'discordance':p_disc
    }


## 3. Partial Spearman controlling transition count

In [ ]:

def safe_spearman(x,y):
    x=np.asarray(x,float)
    y=np.asarray(y,float)
    if len(x)<3 or np.unique(x).size<2 or np.unique(y).size<2:
        return np.nan
    return float(spearmanr(x,y).statistic)

def partial_spearman(full,single,N):
    rho_fs = safe_spearman(full,single)
    rho_fn = safe_spearman(full,N)
    rho_sn = safe_spearman(single,N)

    if not np.all(np.isfinite([rho_fs,rho_fn,rho_sn])):
        return {
            'rho_raw':rho_fs,
            'rho_full_N':rho_fn,
            'rho_single_N':rho_sn,
            'rho_partial_N':np.nan
        }

    den = np.sqrt(
        max(0.0,1-rho_fn**2) *
        max(0.0,1-rho_sn**2)
    )

    rho_partial = (
        (rho_fs-rho_fn*rho_sn)/den
        if den > 1e-12 else np.nan
    )

    return {
        'rho_raw':rho_fs,
        'rho_full_N':rho_fn,
        'rho_single_N':rho_sn,
        'rho_partial_N':rho_partial
    }


## 4. Run all material × branch × temporal-mode analyses

In [ ]:

rows=[]
pair_rows=[]

for (m,branch,mode),g in scores.groupby(
    ['material','branch','temporal_mode'],
    sort=False
):
    for metric in ['H','H_bal']:
        z = matched_climate_table(g,metric)

        if len(z)<5:
            continue

        raw_tau = kendalltau(z.full,z.single).statistic
        ps = partial_spearman(z.full,z.single,z.N)

        base = {
            'material':m,
            'branch':branch,
            'temporal_mode':mode,
            'metric':metric,
            'n_climates':len(z),
            'N_min':int(z.N.min()),
            'N_median':float(z.N.median()),
            'N_max':int(z.N.max()),
            'n_unique_N':int(z.N.nunique()),
            'tau_raw':raw_tau,
            **ps
        }

        # Exact-N primary statistic.
        exact = conditional_pair_metrics(
            z.full,z.single,z.N,rel_caliper=0.0
        )
        base.update({
            'exactN_n_pairs':exact['n_comparable_pairs'],
            'tau_exactN':exact['conditional_tau'],
            'concordance_exactN':exact['concordance'],
            'discordance_exactN':exact['discordance']
        })

        rows.append(base)

        # Sensitivity to N-matching caliper.
        for cal in CALIPERS:
            pm = conditional_pair_metrics(
                z.full,z.single,z.N,rel_caliper=cal
            )
            pair_rows.append({
                'material':m,
                'branch':branch,
                'temporal_mode':mode,
                'metric':metric,
                'N_caliper_relative':cal,
                **pm
            })

controlled = pd.DataFrame(rows)
caliper = pd.DataFrame(pair_rows)

controlled.to_csv(
    OUT/'controlled_N_rank_agreement.csv',
    index=False
)
caliper.to_csv(
    OUT/'controlled_N_pair_caliper_sensitivity.csv',
    index=False
)

display(
    controlled.sort_values(
        ['metric','temporal_mode','branch','material']
    )
)



## 5. Panel summaries

Important: these are descriptive panel summaries, not independent-material inference.

The most informative comparison is:

\[
\tau_{\rm raw}
\quad\text{vs}\quad
\tau_{\rm exactN}
\]

and, secondarily,

\[
\rho_{\rm raw}
\quad\text{vs}\quad
\rho_{\rm partial\,N}.
\]


In [ ]:

def pos_fraction(x):
    x=pd.Series(x).dropna()
    return np.mean(x>0) if len(x) else np.nan

summary = (
    controlled.groupby(
        ['branch','temporal_mode','metric'],
        as_index=False
    )
    .agg(
        n_materials=('material','nunique'),
        median_tau_raw=('tau_raw','median'),
        median_tau_exactN=('tau_exactN','median'),
        median_exactN_pairs=('exactN_n_pairs','median'),
        fraction_exactN_available=('tau_exactN',lambda x:np.mean(pd.notna(x))),
        median_rho_raw=('rho_raw','median'),
        median_rho_partial_N=('rho_partial_N','median'),
        median_rho_full_N=('rho_full_N','median'),
        median_rho_single_N=('rho_single_N','median')
    )
)

summary['delta_tau_exactN_minus_raw'] = (
    summary.median_tau_exactN-summary.median_tau_raw
)
summary['delta_rho_partial_minus_raw'] = (
    summary.median_rho_partial_N-summary.median_rho_raw
)

summary.to_csv(
    OUT/'controlled_N_panel_summary.csv',
    index=False
)

display(summary)


## 6. Caliper sensitivity summary

In [ ]:

caliper_summary = (
    caliper.groupby(
        ['branch','temporal_mode','metric','N_caliper_relative'],
        as_index=False
    )
    .agg(
        n_materials=('material','nunique'),
        median_conditional_tau=('conditional_tau','median'),
        median_concordance=('concordance','median'),
        median_comparable_pairs=('n_comparable_pairs','median'),
        fraction_materials_with_pairs=(
            'n_comparable_pairs',
            lambda x:np.mean(np.asarray(x)>0)
        )
    )
)

caliper_summary.to_csv(
    OUT/'controlled_N_caliper_summary.csv',
    index=False
)

display(caliper_summary)



## 7. Interpretation rules

### Strong preservation after controlling \(N\)
If exact-\(N\) conditional \(\tau\) remains close to raw Kendall \(\tau\), and partial Spearman remains high, then the single–full agreement cannot be explained primarily by transition count.

### Strong occupancy contribution
If raw \(\tau\) is high but exact-\(N\) conditional \(\tau\) collapses toward zero, then the apparent climate-ranking fidelity is largely inherited from differences in \(N\).

### Mixed case
A moderate reduction indicates that occupancy stabilizes part of the ranking while material-specific equilibrium response preserves additional ordering.

### Important caveat
Exact-\(N\) matching can lose many climate pairs if \(N\) is nearly unique for every climate. In that case the 1–5% caliper curves and partial Spearman carry more information than the exact-\(N\) estimate.

This analysis should replace \(H/N\) as the primary occupancy-control sensitivity if the scientific question is specifically whether the **original cumulative \(H\) ranking** survives after controlling occupancy count.
